In [0]:
# COMMAND ----------
# MAGIC %pip install pytest-sugar

# Manually defining the schema to avoid "empty folder" errors
# This ensures the pipeline is robust even if data arrives late
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, TimestampType

# COMMAND ----------
# CONFIGURATION
# Ensure this path contains at least one CSV file before running
source_path = "/FileStore/tables/crude_ops_raw/" 
checkpoint_path = "/mnt/crude_ops/_checkpoints/ingestion"
target_table = "crude_ops.bronze.drilling_raw"


drilling_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("depth", DoubleType(), True),
    StructField("gamma_ray", DoubleType(), True),
    StructField("operation", StringType(), True),
    StructField("phase", StringType(), True)
])

raw_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .schema(drilling_schema) # Manually assigned schema
  .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
  .load(source_path))

# COMMAND ----------
# WRITE STREAM TO DELTA TABLE (BRONZE LAYER)
# Trigger AvailableNow makes it behave like a cost-efficient batch job
query = (raw_stream.writeStream
  .format("delta")
  .option("checkpointLocation", f"{checkpoint_path}/data")
  .option("mergeSchema", "true") # Allows for schema changes over time
  .outputMode("append")
  .trigger(availableNow=True) 
  .toTable(target_table))

query.awaitTermination()